In [ ]:
import pandas as pd
from pathlib import Path
path_to_folder = Path('path_to_your_TMS_folder')
df = pd.read_spss(path_to_folder / 'TMS_data_from_2023.sav')
tot_nb_obs = len(df)
print('Total number of observations:', tot_nb_obs)
df = df.loc[df.Q2B_ST_markets_detail_new2 != 'Switzerland', :]
nb_foreigners = len(df)
print('Number of foreigners in the dataset:', nb_foreigners, '(' + str(100 * round(nb_foreigners / tot_nb_obs, 3)) + '%)')
df = df.loc[~df.Q9_bereinigt_gruppiert_final.isnull(), :]
nb_obs = len(df)
print('Number of observations with main mode choice:', nb_obs, '(' + str(100 * round(nb_obs / nb_foreigners, 3)) + '%)')
df = df.rename(columns={'Q9_bereinigt_gruppiert_final': 'transport_mode',
                        'Q48': 'age',
                        'Q2B_ST_markets_detail_new2': 'country'})

In [ ]:
# Modal split
grouped = df[['transport_mode', 
              'Gewichtung']].groupby(df.transport_mode).apply(lambda x: 
                                                              round(100 * x.sum(numeric_only=True) / df.Gewichtung.sum(), 
                                                                    1))
grouped.rename(columns={'Gewichtung': 'Mode shares'}, inplace=True)
grouped

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
fig, ax = plt.subplots(figsize=(10, 4))
grouped.T.plot(kind='barh', stacked=True, ax=ax, color=['#4e517e',  # Biking
                                                        'r',  # Camper
                                                        '#000000', 
                                                        '#FFF680', 
                                                        'g',  # Long-distance-bus through Europe (with timetable)
                                                        '#000000', 
                                                        '0.8',  # Other
                                                        'b', 
                                                        '#E33B3B'])
#grouped.T.plot.barh(stacked=True, ax=ax)
ax.legend(title='Main transport mode')
sns.move_legend(ax, bbox_to_anchor=(1.01, 1.02), loc='upper left')
ax.set_yticks([])
n=0
for p in ax.patches:
    if n in [2, 3, 8]:
        h, w, x, y = p.get_height(), p.get_width(), p.get_x(), p.get_y()
        text = f'{w:0.0f}%'
        if n == 3:
            text_color = 'black'
        else:
            text_color = 'white'
        ax.annotate(text=text, xy=(x + w / 2, y + h / 2), ha='center', va='center', color=text_color, size=16)
    n = n+1
plt.xlim([0, 100])
plt.title('Modal split of foreign tourists in Switzerland')
plt.tight_layout()

In [ ]:
# Age
age_grouped = df[['age', 
                  'Gewichtung']].groupby(df.age).apply(lambda x: 
                                                       round(100 * x.sum(numeric_only=True) / df.Gewichtung.sum(), 
                                                             1))
age_grouped.rename(columns={'Gewichtung': 'Age distribution'}, inplace=True)
age_grouped

In [ ]:
age_grouped_plot = age_grouped[0:14]
ax = sns.barplot(age_grouped_plot.T)
ax.xaxis.label.set_visible(False)
plt.title('Age distribution of foreign tourists in Switzerland')
ticks_labels = ['16-20', 
                '21-25', 
                '26-30', 
                '31-35', 
                '36-40', 
                '41-45', 
                '46-50', 
                '51-55', 
                '55-60', 
                '61-65', 
                '66-70', 
                '71-75', 
                '76-80', 
                '81-85']
ticks_positions = range(0, len(ticks_labels))
ax.set_xticks(ticks_positions, ticks_list)
ax.set_xticklabels(ticks_list, rotation=45, horizontalalignment='right', rotation_mode='anchor')
ax.get_yaxis().set_visible(False)

In [ ]:
fig, axs = plt.subplots(3, 3)
n=0
for mode in df.transport_mode.unique():
    df_by_mode = df.loc[df['transport_mode'] == mode, ['age', 'Gewichtung']]
    age_grouped_by_mode = df_by_mode[['age', 
                                      'Gewichtung']].groupby(df_by_mode.age).apply(lambda x: 
                                                                                   round(100 * x.sum(numeric_only=True) / df_by_mode.Gewichtung.sum(), 
                                                                                         1))
    age_grouped_by_mode.rename(columns={'Gewichtung': 'Age distribution'}, inplace=True)
    age_grouped_by_mode_plot = age_grouped_by_mode[0:14]
    if n == 0:
        pos = axs[0, 0]
    elif n == 1:
        pos = axs[0, 1]
    elif n == 2:
        pos = axs[0, 2]
    elif n == 3:
        pos = axs[1, 0]
    elif n == 4:
        pos = axs[1, 1]
    elif n == 5:
        pos = axs[1, 2]
    elif n == 6:
        pos = axs[2, 0]
    elif n == 7:
        pos = axs[2, 1]
    elif n == 8:
        pos = axs[2, 2]
    n = n+1
    sns.barplot(age_grouped_by_mode_plot.T, ax=pos)
    pos.get_xaxis().set_visible(False)
    pos.get_yaxis().set_visible(False)
    if mode == 'bicycle, racing bike, mountain bike':
        mode = 'bicycle'
    elif mode == 'long-distance bus through Europe (with timetable)':
        mode = 'long-distance bus'
    elif mode == 'private tour bus (e.g. for group travel)':
        mode = 'private tour bus'
    pos.set_title(mode)

In [ ]:
''' Modal split by country (or group of countries)'''
def modal_split_by_country(list_of_countries, nb_rows, nb_columns, save_figure=False):
    fig, axs = plt.subplots(nb_rows, nb_columns)
    m=0
    for country in list_of_countries:
        df_by_country = df.loc[df['country'] == country, ['transport_mode', 'Gewichtung']]
        grouped_by_country = df_by_country[['transport_mode', 
                                        'Gewichtung']].groupby(df_by_country.transport_mode).apply(lambda x: 
                                                                                                   round(100 * x.sum(numeric_only=True) / df_by_country.Gewichtung.sum(), 
                                                                                                         1))
        grouped_by_country.rename(columns={'Gewichtung':'Mode shares'}, inplace=True)
        if nb_rows == 2 and nb_columns == 2:
            if m == 0:
                pos = axs[0, 0]
            elif m == 1:
                pos = axs[0, 1]
            elif m == 2:
                pos = axs[1, 0]
            elif m == 3:
                pos = axs[1, 1]
        elif nb_rows == 3 and nb_columns == 2:
            if m == 0:
                pos = axs[0, 0]
            elif m == 1:
                pos = axs[0, 1]
            elif m == 2:
                pos = axs[1, 0]
            elif m == 3:
                pos = axs[1, 1]
            elif m == 4:
                pos = axs[2, 0]
            elif m == 5:
                pos = axs[2, 1]
        elif nb_rows == 3 and nb_columns == 3:
            if m == 0:
                pos = axs[0, 0]
            elif m == 1:
                pos = axs[0, 1]
            elif m == 2:
                pos = axs[0, 2]
            elif m == 3:
                pos = axs[1, 0]
            elif m == 4:
                pos = axs[1, 1]
            elif m == 5:
                pos = axs[1, 2]
            elif m == 6:
                pos = axs[2, 0]
            elif m == 7:
                pos = axs[2, 1]
            elif m == 8:
                pos = axs[2, 2]
        elif nb_rows == 4 and nb_columns == 2:
            if m == 0:
                pos = axs[0, 0]
            elif m == 1:
                pos = axs[0, 1]
            elif m == 2:
                pos = axs[1, 0]
            elif m == 3:
                pos = axs[1, 1]
            elif m == 4:
                pos = axs[2, 0]
            elif m == 5:
                pos = axs[2, 1]
            elif m == 6:
                pos = axs[3, 0]
            elif m == 7:
                pos = axs[3, 1]
        elif nb_rows == 3 and nb_columns == 1:
            if m == 0:
                pos = axs[0]
            elif m == 1:
                pos = axs[1]
            elif m == 2:
                pos = axs[2]
        elif nb_rows == 1 and nb_columns == 2:
            if m == 0:
                pos = axs[0]
            elif m == 1:
                pos = axs[1]
        elif nb_rows == 1 and nb_columns == 1:
            pos = axs
        grouped_by_country.T.plot(kind='barh', 
                              stacked=True, 
                              ax=pos, 
                              color=['#4e517e',  # Biking
                                     'r',  # Camper
                                     '#000000', 
                                     '#FFF680', 
                                     'g',  # Long-distance-bus through Europe (with timetable)
                                     '#000000', 
                                     '0.8',  # Other
                                     'b', 
                                     '#E33B3B'])
        pos.get_legend().remove()
        pos.set_yticks([])
        pos.set_xlim([0, 100])
        n=0
        for p in pos.patches:
            if nb_rows == 3 and nb_columns == 3:
                list_modes_with_percentage = [2, 8]
            else:
                list_modes_with_percentage = [2, 3, 8]
            if n in list_modes_with_percentage:
                h, w, x, y = p.get_height(), p.get_width(), p.get_x(), p.get_y()
                text = f'{w:0.0f}%'
                if n == 3:
                    text_color = 'black'
                else:
                    text_color = 'white'
                pos.annotate(text=text, xy=(x + w / 2, y + h / 2), ha='center', va='center', color=text_color, size=9)
            n = n+1
        pos.set_title(country)
        pos.get_xaxis().set_visible(False)
        m = m+1
        pos.annotate(text='Nb obs.: ' + str(len(df_by_country)), xy=(60, -0.37), ha='center', va='center', color='black', size=12)
    plt.tight_layout()
    if save_figure:
        plt.savefig('modal_split_' + list_of_countries[0] + '.png')

In [ ]:
modal_split_by_country(['Greater China',  'India', 'USA', 
                        'United Kingdom', 'Italy', 
                        'Austria', 'Netherlands', 'Germany', 'France'], 3, 3)

In [ ]:
modal_split_by_country(['Spain', 'Netherlands', 'Belgium', 'Denmark', 'United Kingdom', 'Poland'], 3, 2)

In [ ]:
modal_split_by_country(['Sweden', 'Norway', 'Finland'], 3, 1)

In [ ]:
modal_split_by_country(['Czech Republic', 'United Kingdom'], 1, 2)

In [ ]:
modal_split_by_country(['Canada', 'USA',
                        'Australia', 'Brazil',
                        'New Zealand', 'others'], 3, 2)

In [ ]:
modal_split_by_country(['Thailand', 'India',
                        'Singapore', 'Greater China',
                        'Korea', 'Indonesia',
                        'Japan', 'Malaysia'], 4, 2)

In [ ]:
modal_split_by_country(['United Arab Emirates', 'Bahrain',
                        'Saudi Arabia', 'Kuwait',
                        'Qatar', 'Oman'], 3, 2)

In [ ]:
modal_split_by_country(['Russia'], 1, 1)

In [ ]:
# Turn interactive plotting off
plt.ioff()

for unique_country in df.country.unique():
    modal_split_by_country([unique_country], 1, 1, save_figure=True)
    plt.close()

In [ ]:
''' Modal split by region (group of countries) '''
def modal_split_by_region(list_of_countries, title):
    df_by_country = df.loc[df['country'].isin(list_of_countries), ['transport_mode', 'Gewichtung']]
    grouped_by_country = df_by_country[['transport_mode', 
                                    'Gewichtung']].groupby(df_by_country.transport_mode).apply(lambda x: 
                                                                                               round(100 * x.sum(numeric_only=True) / df_by_country.Gewichtung.sum(), 
                                                                                                     1))
    grouped_by_country.rename(columns={'Gewichtung':'Mode shares'}, inplace=True)
    ax = grouped_by_country.T.plot(kind='barh', 
                          stacked=True, 
                          color=['#4e517e',  # Biking
                                 'r',  # Camper
                                 '#000000', 
                                 '#FFF680', 
                                 'g',  # Long-distance-bus through Europe (with timetable)
                                 '#000000', 
                                 '0.8',  # Other
                                 'b', 
                                 '#E33B3B'])
    ax.get_legend().remove()
    ax.set_yticks([])
    ax.set_xlim([0, 100])
    n=0
    for p in ax.patches:
        if n in [2, 3, 8]:
            h, w, x, y = p.get_height(), p.get_width(), p.get_x(), p.get_y()
            text = f'{w:0.0f}%'
            if n == 3:
                text_color = 'black'
            else:
                text_color = 'white'
            ax.annotate(text=text, xy=(x + w / 2, y + h / 2), ha='center', va='center', color=text_color, size=12)
        n = n+1
    ax.set_title(title)
    ax.get_xaxis().set_visible(False)
    ax.annotate(text='Nb obs.: ' + str(len(df_by_country)), xy=(70, -0.37), ha='center', va='center', color='black', size=12)
    plt.tight_layout()

In [ ]:
# Turn interactive plotting off
plt.ion()

modal_split_by_region(['Sweden', 'Norway', 'Finland'], 'Skandinavien')